<a href="https://colab.research.google.com/github/Lateephah/Applied-Search-Intelligence-System/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This notebook trains a real model for my lane (content-refresh review priority) and compares it
to my Week-4 rule baseline **on the same held-out data and the same metric** — precision@K,
because the output is a *ranked queue*, not a plain yes/no call.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Lateephah/Applied-Search-Intelligence-System"
REPO_DIR = "Applied-Search-Intelligence-System"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

print("Working dir:", os.getcwd())

Working dir: /content/Applied-Search-Intelligence-System


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path("data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(RAW_PATH)
initial_rows = len(df)

# Same lane population as w02/w04: real search visibility, old enough to have a trend.
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

base_rate = df["is_declining_label"].mean()
print(f"Rows: {len(df):,} of {initial_rows:,} raw rows")
print(f"Base rate (share declining): {base_rate:.3f}")
print(f"Unique clients: {df['client_id'].nunique()}")

Rows: 30,000 of 30,000 raw rows
Base rate (share declining): 0.542
Unique clients: 32


## 1. Method choice and why

**Task shape:** the output is a *priority queue*, "which pages should an editor look at first?" ,
not a plain yes/no call. From the method table:

| Question shape | Start with |
|---|---|
| yes/no with an observed label | Logistic Regression, then Random Forest |
| "which first?" ranking | any classifier's probability, evaluated at precision@K |

My case is really both rows at once: `is_declining_label` is a binary proxy label (rule-derived
from `trend_direction`, not a directly-observed outcome), but the thing I actually deliver is a
*ranked* list, exactly like the baseline's `score` column. So I train classifiers the normal way,
but I never grade them by accuracy, I sort the held-out set by each model's predicted
probability and score it the same way I scored my rule: **precision@20 / @50 / @100**.

**Models, in order of complexity:**
1. **Logistic Regression** : readable first. I can look at its coefficients and say *why* it
   thinks a page is declining, the same way my rule's reason codes said why a page was flagged.
2. **Random Forest**: a stronger, non-linear model, to see whether the interactions my hand-built
   rule couldn't express (e.g. "staleness matters more when CTR is *also* weak") are worth the
   loss of readability. Per the skill's own rule: *"a depth-2 decision tree you can print and read
   teaches more than an opaque model 2 points stronger. Add complexity only when the comparison
   earns it."* I hold the Random Forest to that bar in section 3.

**Why ML might beat my Week-4 rule here:** the rule uses exactly two hard-coded gates
(`impressions_90d >= 500` and `days_since_last_update >= 90`) plus one binary bonus
(below-tier-median CTR). Everything else I measured in the repo: `word_count`, `engagement_rate`,
`scroll_rate`, `ai_traffic_pct`, `content_type`, `search_volume`, `competition`, never entered
the rule at all, because a human reading bucket tables can only reasonably juggle 2–3 signals at
once. A model can weigh dozens of signals simultaneously and learn *how much* each one matters,
not just whether it clears a threshold I picked by eye.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
print(df.groupby("client_id").size().sort_values(ascending=False).head(10))
print()
print(df.groupby("client_id").size().describe())

client_id
client_19581e27de    7008
client_6208ef0f77    3681
client_4e07408562    2294
client_3fdba35f04    2267
client_f369cb89fc    1796
client_8527a891e2    1194
client_a88a7902cb    1171
client_d4735e3a26    1106
client_7f2253d7e2    1043
client_f74efabef1    1031
dtype: int64

count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
dtype: float64


**Grouped by `client_id`, not a random row split.** There are only 32 clients, and page counts
per client range from 3 to 7,008 (median 567), extremely unequal. Pages from the same client
share editorial style, CMS quirks, template structure, and domain authority. A random row split
would put some of a client's pages in train and others in test, so the model could partly
"memorize" that client's baseline CTR or update cadence rather than learning a signal that
transfers to a *client it has never seen*, which is the realistic situation for any new client
onboarded onto this system. I use `GroupShuffleSplit` (80/20, one split) grouped on `client_id`,
with a fixed `random_state` for reproducibility.

**Not time-aware:** this starter CSV is a single point-in-time snapshot (no `report_date` column
at this grain, that only exists in the full warehouse I queried in w03). There's no way to hold
out "the future" here, so a time-based split isn't available; that's a real limitation of this
dataset, not a choice I'm hiding.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Leakage guard: exclude the columns the label itself is built from, plus IDs.
leakage_cols = [
    "trend_direction", "trend_pct",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "is_declining_label", "content_id", "client_id",
]

numeric_features = [
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "age_tier_order",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "search_volume", "competition", "cpc", "word_count", "char_count",
]
categorical_features = [
    "content_type", "main_intent", "competition_level", "freshness_tier",
    "word_count_tier", "char_count_tier", "impression_tier", "position_tier",
    "provider_used", "model_used",
]

assert set(numeric_features + categorical_features).isdisjoint(leakage_cols)
feature_cols = numeric_features + categorical_features

X = df[feature_cols]
y = df["is_declining_label"]
groups = df["client_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

df_train, df_test = df.iloc[train_idx].reset_index(drop=True), df.iloc[test_idx].reset_index(drop=True)
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train: {len(X_train):,} rows, {df_train['client_id'].nunique()} clients")
print(f"Test:  {len(X_test):,} rows, {df_test['client_id'].nunique()} clients")
print(f"Client overlap between train and test: {set(df_train.client_id) & set(df_test.client_id)}")
print(f"Train decline rate: {y_train.mean():.3f}   Test decline rate: {y_test.mean():.3f}")

Train: 23,837 rows, 25 clients
Test:  6,163 rows, 7 clients
Client overlap between train and test: set()
Train decline rate: 0.550   Test decline rate: 0.511


Zero client overlap, as expected from a group split. Train/test decline rates are close
(both near the 0.54 base rate), so the split isn't accidentally stratifying the label in a
misleading way.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as my Week-4 baseline. Show the table.*

First I recompute my Week-4 rule's score **only on the test rows**,the exact same score
formula from `w04_baseline_score.ipynb` (visibility gate, staleness gate, CTR-gap bonus), so the
baseline row in the comparison table is scored on the identical held-out slice as the models,
not the full 30,000-row population like it was in week 4.

In [ ]:
# --- Rebuild the Week-4 rule, applied only to df_test (same logic, same thresholds) ---
usable_tiers = ["page_1", "page_3_5", "striking"]
pos_known_mask = df_train["avg_position"] > 0
# tier medians are a property of the population the rule was audited on -- computed on TRAIN
# only, then applied to test, so no test information leaks into the rule's own thresholds either.
tier_median_map = (
    df_train.loc[pos_known_mask & df_train["position_tier"].isin(usable_tiers)]
    .groupby("position_tier")["ctr"].median()
)

def score_with_rule(frame):
    frame = frame.copy()
    frame["tier_median_ctr"] = frame["position_tier"].map(tier_median_map)
    visible = frame["impressions_90d"] >= 500
    stale = frame["days_since_last_update"] >= 90
    ctr_gap = (
        (frame["avg_position"] > 0)
        & frame["position_tier"].isin(usable_tiers)
        & (frame["ctr"] < frame["tier_median_ctr"])
    )
    gate = visible & stale
    bonus = np.where(ctr_gap, 0.5, 0.0)
    return np.where(gate, (1 + bonus) * np.log1p(frame["impressions_90d"]), 0.0)

baseline_test_score = score_with_rule(df_test)
print(f"Baseline rule flags {(baseline_test_score > 0).sum():,} of {len(df_test):,} test rows")

Baseline rule flags 575 of 6,163 test rows


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
])

logreg = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=2000, random_state=42)),
])
logreg.fit(X_train, y_train)

rf = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=20,
        random_state=42, n_jobs=-1,
    )),
])
rf.fit(X_train, y_train)

logreg_test_score = logreg.predict_proba(X_test)[:, 1]
rf_test_score = rf.predict_proba(X_test)[:, 1]
print("Both models fit.")

Both models fit.


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-scores)
    top_k_labels = np.asarray(labels)[order][:k]
    return top_k_labels.mean()

y_test_arr = y_test.to_numpy()
test_base_rate = y_test_arr.mean()

rows = []
for name, scores in [
    ("Week-4 rule baseline", baseline_test_score),
    ("Logistic Regression", logreg_test_score),
    ("Random Forest", rf_test_score),
]:
    row = {"model": name}
    for k in [20, 50, 100]:
        row[f"precision@{k}"] = round(precision_at_k(scores, y_test_arr, k), 3)
    row["ROC-AUC"] = round(roc_auc_score(y_test_arr, scores), 3)
    row["Avg Precision"] = round(average_precision_score(y_test_arr, scores), 3)
    rows.append(row)

comparison = pd.DataFrame(rows).set_index("model")
# "No model" reference: with ZERO ranking signal, the expected precision at any K is just
# the base rate, NOT computed via argsort on tied scores, which silently falls back to
# original row order and gives a meaningless number (I caught this as a bug on first run).
comparison.loc["(no-model reference = test base rate)"] = [round(test_base_rate, 3)] * 3 + [np.nan, np.nan]
comparison

,precision@20,precision@50,precision@100,ROC-AUC,Avg Precision
model,,,,,
Week-4 rule baseline,0.500,0.520,0.470,0.489,0.508
Logistic Regression,0.650,0.640,0.670,0.577,0.567
Random Forest,0.600,0.580,0.530,0.602,0.582
(no-model reference = test base rate),0.511,0.511,0.511,NaN,NaN


**Reading the table honestly, this did not go the way my week-4 notebook would have predicted:**

- **The rule baseline underperforms the base rate on this held-out slice** (precision@20=0.500, @50=0.520, @100=0.470, all at or below the 0.511 no-model reference). In week 4, the same rule scored 0.650 at precision@20 -- but that number came from the rule's own tuning population (the full 30,000 rows, including the very clients it was eyeballed against). On 7 *completely unseen* clients, the rule is essentially coin-flip, and on precision@100 it's actually a little worse than doing nothing. That is a genuinely useful, slightly humbling result: **my week-4 rule does not generalize to a new client**, it likely picked up some patterns specific to the clients it was built against.
- **Logistic Regression wins cleanly at every precision@K cut** (0.650 / 0.640 / 0.670);it beats both the rule and the Random Forest at the metric that actually matters for this lane (what's in the top of an editor's queue).
- **Random Forest wins on the whole-ranking measures** (ROC-AUC 0.602 vs Logistic's 0.577; Average Precision 0.582 vs 0.567) but **loses to Logistic Regression specifically at precision@20/50/100**; exactly the "report both, that IS the finding" case the skill warns about. AUC and Average Precision reward getting the *whole* ranking right; my actual job is getting the *top* of the queue right.
- **Applying the skill's own bar**; "add complexity only when the comparison earns it"; Random Forest's extra opacity does **not** earn its keep here. For this lane's actual use case, I'd ship **Logistic Regression**: it's simpler, its coefficients are readable the same way my rule's reason codes were, and it wins on the metric an editor would actually feel.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis with 2–3 concrete cases.*

In [ ]:
from sklearn.inspection import permutation_importance

# Permutation importance on the stronger model, using average precision as the scorer
# (matches the ranking framing, not plain accuracy).
perm = permutation_importance(
    rf, X_test, y_test, scoring="average_precision",
    n_repeats=10, random_state=42, n_jobs=-1,
)
importances = pd.Series(perm.importances_mean, index=feature_cols).sort_values(ascending=False)
print("Top 10 features by permutation importance (Random Forest):")
print(importances.head(10).round(4))

Top 10 features by permutation importance (Random Forest):
days_with_impressions    0.0209
content_age_days         0.0101
impressions_90d          0.0071
ctr                      0.0058
avg_position             0.0042
clicks_90d               0.0040
position_tier            0.0032
impression_tier          0.0025
days_with_sessions       0.0021
scroll_rate              0.0020
dtype: float64


**Do the top features make sense?** (sanity check against leakage, a suspiciously perfect top feature would be a red flag)

None of the top 10 sits anywhere near the importance a leaked column would show (the top score is 0.021, tiny, consistent with a model spreading weight across many weak signals rather than keying on one that secretly encodes the label). But there's a real surprise worth sitting with: **`days_since_last_update`, the one signal my whole week-4 rule was built around, doesn't appear in the top 10 at all.** The model leans hardest on `days_with_impressions` and `content_age_days` instead: how *consistently* a page shows up in search over its lifetime, and how *old* it is, matter more to this model than how long since an editor last touched it. That's a plausible, not-suspicious story (a page's decline is more about its natural traffic lifecycle than about a specific edit date), but it's also a direct, honest challenge to my week-4 rule's core assumption -- worth carrying into week 6.

In [ ]:
# Three concrete wrong cases from the Random Forest, one of each interesting kind:
# a confident false positive, a confident false negative, and a near-the-line miss.
test_view = df_test[["content_id", "client_id", "impressions_90d", "days_since_last_update",
                      "ctr", "avg_position", "position_tier", "content_type",
                      "word_count", "engagement_rate"]].copy()
test_view["true_label"] = y_test_arr
test_view["predicted_prob"] = rf_test_score
test_view["error"] = test_view["predicted_prob"].round() - test_view["true_label"]

false_positives = test_view[(test_view["true_label"] == 0) & (test_view["predicted_prob"] > 0.7)]
false_negatives = test_view[(test_view["true_label"] == 1) & (test_view["predicted_prob"] < 0.3)]

print("Confident false positive (predicted declining, actually stable):")
print(false_positives.sort_values("predicted_prob", ascending=False).head(1).to_string(index=False))
print("\nConfident false negative (predicted stable, actually declining):")
print(false_negatives.sort_values("predicted_prob").head(1).to_string(index=False))

Confident false positive (predicted declining, actually stable):
          content_id         client_id  impressions_90d  days_since_last_update  ctr  avg_position position_tier    content_type  word_count  engagement_rate  true_label  predicted_prob  error
content_c148e44db30d client_8527a891e2              335                     104  0.0          31.3      page_3_5 keyword article      1622.0              0.0           0        0.860266    1.0

Confident false negative (predicted stable, actually declining):
          content_id         client_id  impressions_90d  days_since_last_update  ctr  avg_position position_tier    content_type  word_count  engagement_rate  true_label  predicted_prob  error
content_7bc32bc1df59 client_8527a891e2                1                      92  0.0           0.0         top_3 keyword article      1429.0              0.0           1        0.163028   -1.0


**Why these are hard:**

- **The false positive** (`content_c148e44db30d`) has 335 impressions/90d, notice that's *below* my week-4 rule's own 500-impression visibility gate, so the rule wouldn't even have flagged this page, yet the model is 86% confident it's declining. It has 0.0 CTR and 0.0 engagement on a 1,622-word page ranking at position ~31. My read: this page may be *chronically* low-performing rather than *newly* declining, it never had traffic to lose, so "decline" (a 30-day-vs-prior-30-day comparison) may not even be a meaningful label for it. That's a limit of the label, not obviously a limit of the model.
- **The false negative** (`content_7bc32bc1df59`) has exactly **1 impression** in 90 days and `avg_position = 0.0`, tagged `top_3`, this is the *exact same data trap* I found auditing signals in week 4: `avg_position == 0` means "no position data," not literally ranking #1, and it silently lands in the `top_3` tier. A page with 1 impression total has essentially no reliable trend to measure, yet it's carrying a confident `is_declining_label = 1`. This isn't a new mistake, it's the same "rates need denominators" lesson from my signal audit, now showing up inside the *label itself*, not just inside my rule's inputs.
- **Structural takeaway:** neither error looks like a model bug. Both point at the same place: the 90-day trend label is noisiest exactly where impressions are thinnest, which is worth flagging explicitly for week 6 rather than quietly trusting precision@K on the whole test set.

## Self-check

Before you submit, confirm each line honestly:

- [x] Compares against the baseline on the same split (Week-4 rule re-scored on the test-only slice)
- [x] Uses a valid, honest split design (grouped by `client_id`, zero client overlap, explained why)
- [x] Explains method choice (Logistic Regression → Random Forest, tied to the ranking metric)
- [x] Reports useful metrics (precision@20/50/100, ROC-AUC, Average Precision, all in one table)
- [x] Interprets features (permutation importance) and errors (3 concrete wrong cases)
- [x] Does not reward complexity alone — Random Forest's gain is checked against Logistic
      Regression specifically, not just against the rule
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.